# MB5B — Estoque Diário

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario`
**Transação SAP:** MB5B · **Colunas:** 12
**Clustering declarado:** _(nenhum declarado)_

---

## Objetivo
Mapear o comportamento desta tabela **antes** de qualquer comparação com o SAP.
O resultado alimenta a base de conhecimento do agente de validação e define o cenário de teste.

## Como usar
1. Execute a célula **1** para criar os widgets, depois ajuste os filtros no topo (opcional).
2. Execute a célula **2** — ela cria a view `base`, usada por todas as demais.
3. Execute as células na ordem e leia a coluna **`veredito`** de cada resultado.
4. Exporte o notebook executado para a pasta de conhecimento do agente.

## Aviso metodológico
Contagem de linhas **não** é evidência de qualidade. Erros de colapso de granularidade
preservam o total. Ver seções **4**, **5** e **14**.


## 1. Widgets de recorte

Execute uma vez. Deixe vazio para analisar a base completa.

In [0]:
-- 1. WIDGETS (deixe vazio = sem filtro)
-- Parametros criados via notebook UI (SQL warehouse nao suporta CREATE WIDGET).
-- Referenciados nas demais celulas via :f_cod_centro e :f_cod_empresa
SELECT
  :f_cod_centro AS filtro_centro,
  :f_cod_empresa AS filtro_empresa;

## 2. View `base`

Aplica os filtros dos widgets uma única vez. **Todas** as células seguintes consultam `base`.

In [0]:
-- 2. VIEW BASE (aplica os filtros dos widgets)
CREATE OR REPLACE TEMP VIEW base AS
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario
WHERE (:f_cod_centro = '' OR `cod_centro` = :f_cod_centro)
  AND (:f_cod_empresa = '' OR `cod_empresa` = :f_cod_empresa);
-- Para teste rapido, descomente:
-- LIMIT 1000000;

SELECT COUNT(*) AS linhas_na_base FROM base;

## 3. Metadados e histórico de carga

Formato, particionamento, clustering real e última atualização.
Divergência entre clustering declarado e chave real é o primeiro indício de problema.

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario;

In [0]:
-- Formato, tamanho e particoes (falha se nao for Delta)
SHOW TBLPROPERTIES dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario;

In [0]:
-- Ultimas operacoes de escrita (falha se for view ou nao-Delta)
SHOW TABLE EXTENDED LIKE 'dev_procurement.corp_curated.tbl_ds_pro_mb5b_estoque_diario';

## 4. Granularidade real

`linhas ÷ chaves distintas`. Razão maior que 1,00 significa que existe uma dimensão
adicional multiplicando as linhas — é preciso descobrir **qual** (seção 14).

In [0]:
-- 4. GRANULARIDADE REAL: linhas / chaves distintas
-- Razao > 1,00 significa que existe dimensao adicional multiplicando linhas.
WITH t AS (SELECT COUNT(*) AS total FROM base),
g AS (
SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` FROM base)
UNION ALL
SELECT 'cod_centro + cod_material + dt_estoque' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_centro`, `cod_material`, `dt_estoque` FROM base)
UNION ALL
SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque + sg_unidade_medida' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, `sg_unidade_medida` FROM base)
)
SELECT g.chave, t.total AS linhas, g.combinacoes_distintas,
       ROUND(t.total / g.combinacoes_distintas, 4) AS linhas_por_chave,
       CASE WHEN g.combinacoes_distintas = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade por chave candidata

Quantas combinações se repetem e qual o pior caso.

In [0]:
-- 5. DUPLICIDADE POR CHAVE
SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, COUNT(*) AS qtd FROM base GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_centro + cod_material + dt_estoque' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_centro`, `cod_material`, `dt_estoque`, COUNT(*) AS qtd FROM base GROUP BY `cod_centro`, `cod_material`, `dt_estoque` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_empresa + cod_centro + cod_material + dt_estoque + sg_unidade_medida' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, `sg_unidade_medida`, COUNT(*) AS qtd FROM base GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, `sg_unidade_medida` HAVING COUNT(*) > 1)
ORDER BY chaves_repetidas DESC;

## 6. Varredura de preenchimento — TODAS as colunas

**Seção mais importante do notebook.**

Detecta coluna nunca carregada. Em validação anterior, esta análise revelou 7 colunas
100% nulas no Datalake — uma delas preenchida em **97,9%** dos registros do SAP.
Este erro **não aparece** em teste por amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- 6. PREENCHIMENTO DE TODAS AS COLUNAS
-- Detecta coluna nunca carregada. Secao mais importante do notebook.
WITH t AS (SELECT COUNT(*) AS total FROM base),
perf AS (
  SELECT stack(12,
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'sg_unidade_medida', 'string', COUNT_IF(`sg_unidade_medida` IS NULL), COUNT_IF(`sg_unidade_medida` IS NOT NULL AND lower(trim(`sg_unidade_medida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida`) RLIKE '^0+([.,]0+)?$'),
    'dt_estoque', 'string', COUNT_IF(`dt_estoque` IS NULL), COUNT_IF(`dt_estoque` IS NOT NULL AND lower(trim(`dt_estoque`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_estoque`) RLIKE '^0+([.,]0+)?$'),
    'estoque_inicial', 'double', COUNT_IF(`estoque_inicial` IS NULL), 0L, COUNT_IF(`estoque_inicial` = 0),
    'entrada', 'double', COUNT_IF(`entrada` IS NULL), 0L, COUNT_IF(`entrada` = 0),
    'saida', 'double', COUNT_IF(`saida` IS NULL), 0L, COUNT_IF(`saida` = 0),
    'estoque_final', 'double', COUNT_IF(`estoque_final` IS NULL), 0L, COUNT_IF(`estoque_final` = 0),
    'dh_carga', 'timestamp', COUNT_IF(`dh_carga` IS NULL), 0L, 0L,
    'dh_atualizacao', 'timestamp', COUNT_IF(`dh_atualizacao` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM base
)
SELECT
  p.coluna,
  p.tipo,
  p.nulos,
  p.vazios,
  p.zeros,
  t.total - p.nulos - p.vazios - p.zeros                              AS uteis,
  ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
  CASE
    WHEN p.nulos = t.total                                      THEN '1. 100% NULO'
    WHEN t.total - p.nulos - p.vazios - p.zeros <= 0            THEN '2. SEM VALOR UTIL'
    WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO (<1%)'
    ELSE '9. ok'
  END AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Valores distintos por coluna. Coluna constante é candidata a default de carga.

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM base),
card AS (
  SELECT stack(12,
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'sg_unidade_medida', 'string', approx_count_distinct(`sg_unidade_medida`),
    'dt_estoque', 'string', approx_count_distinct(`dt_estoque`),
    'estoque_inicial', 'double', approx_count_distinct(`estoque_inicial`),
    'entrada', 'double', approx_count_distinct(`entrada`),
    'saida', 'double', approx_count_distinct(`saida`),
    'estoque_final', 'double', approx_count_distinct(`estoque_final`),
    'dh_carga', 'timestamp', approx_count_distinct(`dh_carga`),
    'dh_atualizacao', 'timestamp', approx_count_distinct(`dh_atualizacao`)
  ) AS (coluna, tipo, distintos)
  FROM base
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE
         WHEN c.distintos <= 1                   THEN '1. CONSTANTE (1 valor)'
         WHEN c.distintos <= 3                   THEN '2. cardinalidade muito baixa'
         WHEN c.distintos > t.total * 0.95       THEN '3. candidata a identificador'
         ELSE '9. normal'
       END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada coluna. Um valor concentrando mais de 99% da base
indica possível default de carga em vez de dado real.

In [0]:
-- 8. DOMINIO DAS COLUNAS CATEGORICAS (top 8 de cada)
-- Valor concentrando >99% indica possivel default de carga.
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida' AS coluna, CAST(`sg_unidade_medida` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `sg_unidade_medida` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

**Atenção ao tipo:** quantidade costuma usar `decimal`, mas valor monetário
frequentemente usa `double` — risco de arredondamento na conciliação financeira.

In [0]:
-- 9. PERFIL DOS CAMPOS NUMERICOS
-- Tipo DOUBLE em valor monetario = risco de arredondamento na conciliacao.
SELECT * FROM (
  SELECT stack(4,
    'estoque_inicial', 'double', COUNT(`estoque_inicial`), CAST(MIN(`estoque_inicial`) AS DOUBLE), CAST(MAX(`estoque_inicial`) AS DOUBLE), CAST(AVG(`estoque_inicial`) AS DOUBLE), CAST(percentile_approx(`estoque_inicial`, 0.5) AS DOUBLE), CAST(percentile_approx(`estoque_inicial`, 0.95) AS DOUBLE), COUNT_IF(`estoque_inicial` < 0), COUNT_IF(`estoque_inicial` = 0),
    'entrada', 'double', COUNT(`entrada`), CAST(MIN(`entrada`) AS DOUBLE), CAST(MAX(`entrada`) AS DOUBLE), CAST(AVG(`entrada`) AS DOUBLE), CAST(percentile_approx(`entrada`, 0.5) AS DOUBLE), CAST(percentile_approx(`entrada`, 0.95) AS DOUBLE), COUNT_IF(`entrada` < 0), COUNT_IF(`entrada` = 0),
    'saida', 'double', COUNT(`saida`), CAST(MIN(`saida`) AS DOUBLE), CAST(MAX(`saida`) AS DOUBLE), CAST(AVG(`saida`) AS DOUBLE), CAST(percentile_approx(`saida`, 0.5) AS DOUBLE), CAST(percentile_approx(`saida`, 0.95) AS DOUBLE), COUNT_IF(`saida` < 0), COUNT_IF(`saida` = 0),
    'estoque_final', 'double', COUNT(`estoque_final`), CAST(MIN(`estoque_final`) AS DOUBLE), CAST(MAX(`estoque_final`) AS DOUBLE), CAST(AVG(`estoque_final`) AS DOUBLE), CAST(percentile_approx(`estoque_final`, 0.5) AS DOUBLE), CAST(percentile_approx(`estoque_final`, 0.95) AS DOUBLE), COUNT_IF(`estoque_final` < 0), COUNT_IF(`estoque_final` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, p95, negativos, zeros)
  FROM base
)
ORDER BY coluna;

## 10. Datas armazenadas como STRING

**Armadilha conhecida:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
É a mesma data em formato diferente — já gerou **16.773 falsos positivos**.

Se a coluna `veredito` acusar mais de um formato, a normalização é obrigatória.

In [0]:
-- 10. DATAS ARMAZENADAS COMO STRING
-- ARMADILHA: SAP exporta '2024-02-23 00:00:00', Datalake grava '20240223'.
-- Mesma data, formato diferente. Ja gerou 16.773 falsos positivos.
WITH t AS (SELECT COUNT(*) AS total FROM base),
d AS (
  SELECT stack(1,
    'dt_estoque', COUNT_IF(`dt_estoque` IS NULL OR trim(`dt_estoque`) = ''), COUNT_IF(trim(`dt_estoque`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_estoque`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_estoque`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_estoque`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_estoque`) NOT IN ('', '00000000') THEN `dt_estoque` END), MAX(CASE WHEN trim(`dt_estoque`) NOT IN ('', '00000000') THEN `dt_estoque` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo)
  FROM base
)
SELECT d.coluna, d.vazios, d.fmt_AAAAMMDD, d.fmt_ISO, d.fmt_BR, d.data_zero,
       t.total - d.vazios - d.fmt_AAAAMMDD - d.fmt_ISO - d.fmt_BR AS nao_reconhecido,
       d.minimo, d.maximo,
       CASE WHEN (CASE WHEN d.fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato na mesma coluna'
            ELSE 'formato unico' END AS veredito
FROM d CROSS JOIN t
ORDER BY d.coluna;

## 11. Códigos — zeros à esquerda, espaços e formato

**Armadilha conhecida:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá **0% de match**.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- 11. CODIGOS: ZEROS A ESQUERDA, ESPACOS E FORMATO
-- ARMADILHA: SAP grava '425263', Datalake grava '000000000000425263'.
-- Sem normalizar, o join da 0% de match.
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes_ao_remover_zeros,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'tem zeros a esquerda' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN com_espacos > 0 THEN 'tem espacos' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END,
         CASE WHEN tipo LIKE '%int%' OR tipo LIKE 'big%'
              THEN 'TIPO NUMERICO - zeros a esquerda JA perdidos' END
       ) AS alertas
FROM (
  SELECT stack(3,
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_material` AS STRING) <> trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_centro` AS STRING) <> trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_empresa', 'string', COUNT_IF(CAST(`cod_empresa` AS STRING) IS NULL OR trim(CAST(`cod_empresa` AS STRING)) = ''), MIN(length(trim(CAST(`cod_empresa` AS STRING)))), MAX(length(trim(CAST(`cod_empresa` AS STRING)))), COUNT_IF(trim(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_empresa` AS STRING) <> trim(CAST(`cod_empresa` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_empresa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
        distintos_bruto, distintos_sem_zeros)
  FROM base
)
ORDER BY coluna;

## 12. Amostra de linhas completas

O dado como ele realmente está: formato de código, decimais, datas e nulos.

In [0]:
-- 12. AMOSTRA
SELECT * FROM base LIMIT 20;

In [0]:
-- 12.1 AMOSTRA ALEATORIA
SELECT * FROM base ORDER BY rand() LIMIT 10;

## 13. Distribuição por dimensão de recorte

Base para escolher o cenário de teste: volume viável (10 mil a 300 mil linhas)
contendo os casos-limite identificados nas seções anteriores.

In [0]:
-- 13. DISTRIBUICAO POR cod_centro
SELECT `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_centro`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_empresa
SELECT `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_empresa`
ORDER BY linhas DESC
LIMIT 40;

## 14. Duplicidade — o que diferencia as linhas repetidas?

Analisando pela chave **cod_empresa + cod_centro + cod_material + dt_estoque**.

**Regra crítica:** linhas **idênticas** = duplicata real (erro de carga).
Linhas **distintas** = granularidade adicional legítima (split valuation, lote, tipo de avaliação).

São problemas diferentes com tratamentos diferentes. Em validação anterior, 5 linhas do mesmo
material eram todas distintas, diferenciadas por um campo que sequer existia nos extratos do SAP.

In [0]:
-- 14. CHAVES DUPLICADAS
SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, COUNT(*) AS qtd
FROM base
GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
-- 14.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS?
-- REGRA: linhas identicas = duplicata real (erro de carga).
--        linhas distintas = granularidade adicional legitima (split valuation, lote...).
WITH dup AS (
  SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` FROM base GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` HAVING COUNT(*) > 1
),
d AS (
  SELECT b.* FROM base b JOIN dup USING (`cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`)
),
agg AS (
  SELECT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `sg_unidade_medida`) AS `sg_unidade_medida`,
         COUNT(DISTINCT `estoque_inicial`) AS `estoque_inicial`,
         COUNT(DISTINCT `entrada`) AS `entrada`,
         COUNT(DISTINCT `saida`) AS `saida`,
         COUNT(DISTINCT `estoque_final`) AS `estoque_final`,
         COUNT(DISTINCT `dh_carga`) AS `dh_carga`,
         COUNT(DISTINCT `dh_atualizacao`) AS `dh_atualizacao`
  FROM d GROUP BY `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1
            THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(8,
    'desc_material', MAX(`desc_material`),
    'sg_unidade_medida', MAX(`sg_unidade_medida`),
    'estoque_inicial', MAX(`estoque_inicial`),
    'entrada', MAX(`entrada`),
    'saida', MAX(`saida`),
    'estoque_final', MAX(`estoque_final`),
    'dh_carga', MAX(`dh_carga`),
    'dh_atualizacao', MAX(`dh_atualizacao`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 15. Freshness — atualidade da carga

In [0]:
-- 15. FRESHNESS
-- DESCRIBE HISTORY nao funciona pois o objeto e uma VIEW.
-- Usando dh_carga e dh_atualizacao como indicadores de freshness.
SELECT
  MAX(dh_carga) AS ultima_carga,
  MAX(dh_atualizacao) AS ultima_atualizacao,
  DATEDIFF(current_timestamp(), MAX(dh_carga)) AS dias_desde_carga,
  DATEDIFF(current_timestamp(), MAX(dh_atualizacao)) AS dias_desde_atualizacao,
  COUNT(DISTINCT DATE(dh_carga)) AS dias_distintos_carga
FROM base;

## 16. Análises específicas — MB5B

### 16.1 Equação contábil do estoque
`estoque_inicial + entrada − saída = estoque_final`

Validação mais importante desta tabela. Desvio indica erro de cálculo na carga
ou perda de movimento.

In [0]:
-- 16.1 EQUACAO CONTABIL
WITH d AS (
  SELECT *,
         estoque_inicial + entrada - saida AS calculado,
         ABS((estoque_inicial + entrada - saida) - estoque_final) AS diferenca
  FROM base
)
SELECT COUNT(*) AS linhas,
       COUNT_IF(diferenca <= 0.001) AS equacao_ok,
       COUNT_IF(diferenca > 0.001) AS divergentes,
       ROUND(100.0 * COUNT_IF(diferenca > 0.001) / COUNT(*), 4) AS pct_divergente,
       MAX(diferenca) AS maior_diferenca,
       COUNT_IF(estoque_inicial IS NULL OR entrada IS NULL
             OR saida IS NULL OR estoque_final IS NULL) AS com_campo_nulo,
       CASE WHEN COUNT_IF(diferenca > 0.001) = 0
            THEN 'OK - equacao fecha em todas as linhas'
            ELSE 'ERRO - investigar antes de comparar com o SAP' END AS veredito
FROM d;

In [0]:
-- 16.1b PIORES DIVERGENCIAS DA EQUACAO
SELECT cod_empresa, cod_centro, cod_material, dt_estoque,
       estoque_inicial, entrada, saida, estoque_final,
       estoque_inicial + entrada - saida AS calculado,
       ABS((estoque_inicial + entrada - saida) - estoque_final) AS diferenca
FROM base
WHERE ABS((estoque_inicial + entrada - saida) - estoque_final) > 0.001
ORDER BY diferenca DESC
LIMIT 20;

### 16.2 Continuidade da série temporal
O `estoque_final` de um dia deve ser igual ao `estoque_inicial` do dia seguinte
para o mesmo material + centro. Quebra indica lacuna na carga.

In [0]:
-- 16.2 CONTINUIDADE DA SERIE
WITH d AS (
  SELECT cod_empresa, cod_centro, cod_material, estoque_inicial, estoque_final,
         COALESCE(to_date(CASE WHEN dt_estoque RLIKE '^[0-9]{8}$' THEN dt_estoque END, 'yyyyMMdd'),
                  to_date(dt_estoque)) AS dt
  FROM base
),
w AS (
  SELECT *,
         LAG(estoque_final) OVER (PARTITION BY cod_empresa, cod_centro, cod_material ORDER BY dt) AS final_anterior,
         LAG(dt)            OVER (PARTITION BY cod_empresa, cod_centro, cod_material ORDER BY dt) AS dt_anterior
  FROM d
)
SELECT COUNT_IF(final_anterior IS NOT NULL) AS transicoes,
       COUNT_IF(final_anterior IS NOT NULL AND ABS(estoque_inicial - final_anterior) <= 0.001) AS continuas,
       COUNT_IF(final_anterior IS NOT NULL AND ABS(estoque_inicial - final_anterior) > 0.001) AS quebradas,
       COUNT_IF(datediff(dt, dt_anterior) > 1) AS com_lacuna_de_dias,
       MAX(datediff(dt, dt_anterior)) AS maior_lacuna_dias
FROM w;

In [0]:
-- 16.3 COBERTURA TEMPORAL
WITH d AS (
  SELECT COALESCE(to_date(CASE WHEN dt_estoque RLIKE '^[0-9]{8}$' THEN dt_estoque END, 'yyyyMMdd'),
                  to_date(dt_estoque)) AS dt
  FROM base
)
SELECT MIN(dt) AS primeira_data, MAX(dt) AS ultima_data,
       COUNT(DISTINCT dt) AS dias_com_dado,
       datediff(MAX(dt), MIN(dt)) + 1 AS dias_do_periodo,
       datediff(MAX(dt), MIN(dt)) + 1 - COUNT(DISTINCT dt) AS dias_sem_dado
FROM d;

## 90. Integridade referencial cruzada _(opcional)_

Confere se os códigos desta tabela existem nas tabelas de referência.
Execute apenas se as outras tabelas estiverem acessíveis no mesmo ambiente.

In [0]:
-- 90. INTEGRIDADE: cod_material -> dev_procurement.corp_curated.tbl_ds_mdm_mm60.cod_material
WITH loc AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM base WHERE `cod_material` IS NOT NULL AND trim(CAST(`cod_material` AS STRING)) <> ''
),
ref AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
)
SELECT (SELECT COUNT(*) FROM loc) AS codigos_distintos_aqui,
       (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k)) AS sem_correspondencia,
       ROUND(100.0 * (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k))
                   / (SELECT COUNT(*) FROM loc), 2) AS pct_orfao;

## 99. Resumo consolidado

**Copie a saída desta célula** para o relatório ou para a base de conhecimento do agente.

In [0]:
-- 99. RESUMO CONSOLIDADO
SELECT 'VOLUMETRIA' AS bloco, 'linhas na base' AS item,
       CAST(COUNT(*) AS STRING) AS valor, '' AS veredito
  FROM base
UNION ALL
SELECT 'VOLUMETRIA', 'colunas', '12', ''
UNION ALL
SELECT 'VOLUMETRIA', 'clustering declarado',
       '(nenhum)', ''
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_empresa + cod_centro + cod_material + dt_estoque' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_centro + cod_material + dt_estoque' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_centro`, `cod_material`, `dt_estoque` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_empresa + cod_centro + cod_material + dt_estoque + sg_unidade_medida' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_empresa`, `cod_centro`, `cod_material`, `dt_estoque`, `sg_unidade_medida` FROM base)
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 13**.
2. Extrair a transação no SAP com o **mesmo recorte** e na **mesma data** do snapshot.
3. Extrair **todas** as abas/telas da transação — comparar parcialmente esconde erros de granularidade.
4. Submeter os arquivos ao agente de validação junto com este notebook executado.

### Checklist antes de comparar com o SAP

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 11)
- [ ] Formato de data normalizado para `AAAAMMDD` (seção 10)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 9)
- [ ] Duplicidades classificadas: idênticas vs granularidade legítima (seção 14)
